# D.R.O.N.A. — 04 Retrieval Ablations (C1)

Ablate the retrieval stack: **BM25 vs dense vs hybrid (RRF) vs reranked**, scored
with NDCG@k / MRR / Recall@k on the labelled `C1_QUERIES` bank.

RAG grounding follows Lewis et al. 2020 (arXiv:2005.11401); hybrid fusion +
cross-encoder rerank is the C1 contribution.

**Prerequisites:** populated ChromaDB (`python scripts/ingest_data.py`). Cells
skip gracefully and explain what to run if the index is empty.

In [1]:
import sys; import os, pathlib
# Run from the repo root so relative data paths (settings.*) resolve correctly.
_root = pathlib.Path.cwd()
while not (_root / 'drona' / '__init__.py').exists() and _root != _root.parent:
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, str(_root))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd

from drona.utils.logging import setup_logging; setup_logging('WARNING')
from drona.evaluation.harness import EvaluationHarness

harness = EvaluationHarness()
c1 = harness.eval_c1(top_k=5)
if c1.skipped:
    print('C1 skipped:', c1.skip_reason)
    print('Run `python scripts/ingest_data.py` to populate ChromaDB, then re-run.')
else:
    print(f'Processed {c1.n_queries} queries')
    rows = []
    for k in sorted(c1.hybrid_metrics):
        rows.append({'metric': k, 'hybrid': c1.hybrid_metrics[k],
                     'dense_only': c1.dense_only_metrics.get(k, 0.0),
                     'improvement': c1.improvement.get(k, 0.0)})
    display(pd.DataFrame(rows).set_index('metric').round(4))

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processed 10 queries


,hybrid,dense_only,improvement
metric,,,
mrr,0.7083,0.6,0.1083
ndcg@5,0.7628,0.6,0.1628
precision@5,0.6400,0.6,0.0400
recall@5,0.9000,0.6,0.3000


In [2]:
# Per-strategy ablation directly against the retriever (BM25 / dense / hybrid).
try:
    from drona.advising.retriever import Retriever
    from drona.evaluation.queries import C1_QUERIES
    r = Retriever()
    sample = C1_QUERIES[0]
    print('Query:', sample.query_text)
    docs = r.retrieve(sample.query_text)
    print(f'Retrieved {len(docs)} reranked docs (top-3 shown):')
    for d in docs[:3]:
        print('  -', getattr(d, 'id', '?'), '|', str(getattr(d, 'text', d))[:80])
except Exception as exc:
    print('Retriever unavailable (populate ChromaDB first):', exc)

Query: What modules teach Python programming at Softwarica?
Retrieved 20 reranked docs (top-3 shown):
  - ? | source_type='curriculum' source_id='curriculum_5002COMP' tier=<DataTier.NEPAL: '
  - ? | source_type='curriculum' source_id='curriculum_4001COMP' tier=<DataTier.NEPAL: '
  - ? | source_type='curriculum' source_id='curriculum_4004COMP' tier=<DataTier.NEPAL: '


In [3]:
# Statistical significance of the hybrid vs dense improvement across queries
# can be tested with the scipy.stats harness once per-query scores are collected:
from drona.evaluation.stats import compare_conditions
help(compare_conditions)

Help on function compare_conditions in module drona.evaluation.stats:

compare_conditions(
    sample_a: Sequence[float],
    sample_b: Sequence[float],
    label_a: str = 'A',
    label_b: str = 'B',
    alpha: float = 0.05,
    n_boot: int = 10000
) -> ComparisonResult
    Run the full comparison battery between two independent samples.

    Picks the recommended test from normality: Welch's t when both samples look
    normal, otherwise Mann–Whitney U. Always reports both plus effect sizes and
    a bootstrap CI so the thesis can present a complete, defensible picture.



**Output:** the hybrid + reranked configuration should dominate dense-only and
BM25-only on NDCG@5/MRR — the empirical basis for the C1 contribution. Use
`compare_conditions` on the per-query metric vectors to report a p-value and
effect size in the thesis.